<a href="https://colab.research.google.com/github/natdanaiii/Trading/blob/main/Grid_trading_V0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# V0-B Initialized Fixed Grid Backtest

Fixed arithmetic spot grid with grid-consistent initial BTC sizing.

**Trading rules:** SELL before BUY; BUY only on downward crossing; new BUY cannot SELL in the same candle; a sold grid cannot rebuy in the same candle; same-candle SELL proceeds cannot fund BUYs; no compounding.

This notebook is for backtest / shadow-readiness only. It does not send Binance orders.


## 0. Setup


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import base64
import bisect
import heapq
import json
import os
import time

import numpy as np
import pandas as pd
import requests


# 1. Trading System

Trading configuration and engine are kept together here. This is the frozen V0-B baseline for later Shadow/Live reuse.


## 1.1 Trading Configuration


In [ ]:
SYMBOL = "BTCUSDT"
INITIAL_CAPITAL = 3000.0

GRID_FLOOR = 38000.0
GRID_CEILING = 127000.0
GRID_GAP = 1000.0

BUY_FEE = 0.001
SELL_FEE = 0.001


## 1.2 Grid & Initialization


In [ ]:
def validate_trading_config(capital, floor, ceiling, gap, buy_fee, sell_fee):
    if capital <= 0 or floor <= 0 or ceiling <= floor or gap <= 0:
        raise ValueError("Invalid trading configuration.")
    if not (0 <= buy_fee < 1 and 0 <= sell_fee < 1):
        raise ValueError("Fees must be in [0, 1).")

    raw_count = (ceiling - floor) / gap
    if not np.isclose(raw_count, round(raw_count)):
        raise ValueError("Grid range must be exactly divisible by GRID_GAP.")

    return int(round(raw_count))


def build_grid_template(floor, ceiling, gap):
    buy_prices = np.arange(floor, ceiling, gap, dtype=float)
    return pd.DataFrame({
        "grid_id": np.arange(1, len(buy_prices) + 1),
        "buy_price": buy_prices,
        "sell_target": buy_prices + gap,
    })


def derive_initialized_grid(grid_template, start_price, initial_capital, buy_fee):
    grid = grid_template.copy()
    floor = float(grid["buy_price"].min())
    ceiling = float(grid["sell_target"].max())

    if not (floor < start_price < ceiling):
        raise ValueError("Starting market price must be inside the grid range.")

    seed_mask = grid["sell_target"] > start_price
    reserve_mask = ~seed_mask
    seed_buy_prices = grid.loc[seed_mask, "buy_price"].to_numpy(float)

    funding_weight = (
        int(reserve_mask.sum())
        + float(np.sum(start_price / seed_buy_prices))
    )
    normal_order_size = initial_capital / funding_weight

    grid["order_size_usdt"] = normal_order_size
    grid["seed_at_start"] = seed_mask
    grid["normal_net_btc"] = (
        normal_order_size / grid["buy_price"] * (1.0 - buy_fee)
    )
    grid["initial_entry_cost_usdt"] = np.where(
        seed_mask,
        grid["normal_net_btc"] / (1.0 - buy_fee) * start_price,
        0.0,
    )

    reserved_cash = float(reserve_mask.sum() * normal_order_size)
    seeded_btc_cost = float(grid["initial_entry_cost_usdt"].sum())

    if not np.isclose(
        reserved_cash + seeded_btc_cost,
        initial_capital,
        atol=1e-8,
    ):
        raise AssertionError("Initialization does not reconcile to capital.")

    return grid, {
        "normal_order_size_usdt": float(normal_order_size),
        "funding_weight": float(funding_weight),
        "initial_sell_positions": int(seed_mask.sum()),
        "initial_buy_levels": int(reserve_mask.sum()),
        "reserved_cash_usdt": reserved_cash,
        "seeded_btc_cost_usdt": seeded_btc_cost,
    }


NUMBER_OF_GRIDS = validate_trading_config(
    INITIAL_CAPITAL,
    GRID_FLOOR,
    GRID_CEILING,
    GRID_GAP,
    BUY_FEE,
    SELL_FEE,
)
GRID_TEMPLATE = build_grid_template(GRID_FLOOR, GRID_CEILING, GRID_GAP)

print(f"Number of Grids : {NUMBER_OF_GRIDS}")


## 1.3 Position & Trading Engine


In [ ]:
def create_position(
    trade_id,
    grid_index,
    buy_time,
    market_buy_price,
    grid_buy_price,
    sell_target,
    normal_order_size,
    portfolio_value_at_buy,
    buy_fee,
    sell_fee,
    entry_type,
):
    if entry_type == "INITIAL_SEED":
        target_net_btc = (
            normal_order_size / grid_buy_price * (1.0 - buy_fee)
        )
        gross_btc = target_net_btc / (1.0 - buy_fee)
        order_size_usdt = gross_btc * market_buy_price
    else:
        order_size_usdt = normal_order_size
        gross_btc = order_size_usdt / market_buy_price

    buy_fee_btc = gross_btc * buy_fee
    btc_amount = gross_btc - buy_fee_btc
    gross_sell_usdt = btc_amount * sell_target
    sell_fee_usdt = gross_sell_usdt * sell_fee

    return {
        "trade_id": int(trade_id),
        "grid_index": int(grid_index),
        "entry_type": entry_type,
        "status": "OPEN",
        "buy_time": buy_time,
        "buy_price": float(market_buy_price),
        "sell_target": float(sell_target),
        "sell_time": pd.NaT,
        "order_size_usdt": float(order_size_usdt),
        "portfolio_value_at_buy": float(portfolio_value_at_buy),
        "order_pct_of_portfolio": float(
            order_size_usdt / portfolio_value_at_buy
        ),
        "portfolio_value_at_sell": np.nan,
        "btc_amount": float(btc_amount),
        "buy_fee_btc": float(buy_fee_btc),
        "sell_fee_usdt": float(sell_fee_usdt),
        "net_sell_usdt": float(gross_sell_usdt - sell_fee_usdt),
        "net_pnl": np.nan,
    }


def initialize_portfolio(data, grid_template, initial_capital, buy_fee, sell_fee):
    start_time = data.iloc[0]["open_time"]
    start_price = float(data.iloc[0]["open"])
    grid, sizing = derive_initialized_grid(
        grid_template, start_price, initial_capital, buy_fee
    )

    positions = {}
    active_by_grid = {}
    sell_heap = []
    events = []
    cash = float(initial_capital)
    btc = 0.0
    total_buy_fee_usdt = 0.0
    trade_id = 0
    event_id = 0

    for grid_index in grid.index[grid["seed_at_start"]]:
        row = grid.loc[grid_index]
        portfolio_value = cash + btc * start_price
        trade_id += 1

        position = create_position(
            trade_id=trade_id,
            grid_index=grid_index,
            buy_time=start_time,
            market_buy_price=start_price,
            grid_buy_price=float(row["buy_price"]),
            sell_target=float(row["sell_target"]),
            normal_order_size=float(row["order_size_usdt"]),
            portfolio_value_at_buy=portfolio_value,
            buy_fee=buy_fee,
            sell_fee=sell_fee,
            entry_type="INITIAL_SEED",
        )

        cash_before, btc_before = cash, btc
        cash -= position["order_size_usdt"]
        btc += position["btc_amount"]
        total_buy_fee_usdt += position["buy_fee_btc"] * start_price

        positions[trade_id] = position
        active_by_grid[grid_index] = trade_id
        heapq.heappush(sell_heap, (position["sell_target"], trade_id))

        event_id += 1
        events.append({
            "event_id": event_id,
            "time": start_time,
            "side": "BUY",
            "trade_id": trade_id,
            "grid_index": grid_index,
            "price": start_price,
            "cash_movement": -position["order_size_usdt"],
            "grid_cashflow": 0.0,
            "cash_before": cash_before,
            "cash_after": cash,
            "btc_before": btc_before,
            "btc_after": btc,
            "portfolio_value_before": portfolio_value,
            "initialization_trade": True,
        })

    initialization = {
        **sizing,
        "start_time": start_time,
        "start_price": start_price,
        "initial_cash": float(cash),
        "initial_btc": float(btc),
        "initial_btc_cost_usdt": sizing["seeded_btc_cost_usdt"],
        "initial_buy_fee_usdt": float(total_buy_fee_usdt),
        "initial_btc_allocation_pct": float(
            sizing["seeded_btc_cost_usdt"] / initial_capital * 100.0
        ),
    }

    return {
        "grid": grid,
        "cash": cash,
        "btc": btc,
        "positions": positions,
        "active_by_grid": active_by_grid,
        "sell_heap": sell_heap,
        "events": events,
        "trade_id": trade_id,
        "event_id": event_id,
        "initialization": initialization,
        "total_buy_fee_usdt": total_buy_fee_usdt,
    }


def run_initialized_fixed_grid(
    data,
    grid_template,
    initial_capital,
    buy_fee,
    sell_fee,
):
    data = data.sort_values("open_time").reset_index(drop=True).copy()
    if data.empty:
        raise ValueError("Market data is empty.")

    state = initialize_portfolio(
        data, grid_template, initial_capital, buy_fee, sell_fee
    )

    grid = state["grid"]
    cash = state["cash"]
    btc = state["btc"]
    positions = state["positions"]
    active_by_grid = state["active_by_grid"]
    sell_heap = state["sell_heap"]
    events = state["events"]
    trade_id = state["trade_id"]
    event_id = state["event_id"]
    initialization = state["initialization"]

    normal_order_size = float(initialization["normal_order_size_usdt"])
    total_buy_fee_usdt = float(state["total_buy_fee_usdt"])
    total_sell_fee_usdt = 0.0
    realized_profit = 0.0
    completed_cycles = 0

    buy_prices = grid["buy_price"].to_numpy(float)
    sell_targets = grid["sell_target"].to_numpy(float)
    buy_price_list = buy_prices.tolist()

    equity_values = np.empty(len(data))
    cash_values = np.empty(len(data))
    btc_values = np.empty(len(data))
    previous_close = None
    tolerance = 1e-12

    for row_index, candle in enumerate(data.itertuples(index=False)):
        timestamp = candle.open_time
        open_price = float(candle.open)
        high_price = float(candle.high)
        low_price = float(candle.low)
        close_price = float(candle.close)

        cash_at_candle_start = cash
        sold_this_candle = set()

        # SELL existing positions first.
        while sell_heap and sell_heap[0][0] <= high_price + tolerance:
            _, current_trade_id = heapq.heappop(sell_heap)
            position = positions.get(current_trade_id)

            if position is None or position["status"] != "OPEN":
                continue

            grid_index = position["grid_index"]
            cash_before, btc_before = cash, btc
            portfolio_value = (
                cash_before + btc_before * position["sell_target"]
            )

            cash += position["net_sell_usdt"]
            btc -= position["btc_amount"]
            if abs(btc) < 1e-12:
                btc = 0.0

            pnl = position["net_sell_usdt"] - position["order_size_usdt"]
            position.update(
                status="CLOSED",
                sell_time=timestamp,
                portfolio_value_at_sell=portfolio_value,
                net_pnl=float(pnl),
            )
            active_by_grid.pop(grid_index, None)

            total_sell_fee_usdt += position["sell_fee_usdt"]
            realized_profit += pnl
            completed_cycles += 1
            sold_this_candle.add(grid_index)

            event_id += 1
            events.append({
                "event_id": event_id,
                "time": timestamp,
                "side": "SELL",
                "trade_id": current_trade_id,
                "grid_index": grid_index,
                "price": position["sell_target"],
                "cash_movement": position["net_sell_usdt"],
                "grid_cashflow": pnl,
                "cash_before": cash_before,
                "cash_after": cash,
                "btc_before": btc_before,
                "btc_after": btc,
                "portfolio_value_before": portfolio_value,
                "initialization_trade": False,
            })

        # BUY only on downward crossings.
        buy_budget = cash_at_candle_start
        downward_start = (
            open_price
            if previous_close is None
            else max(previous_close, open_price)
        )

        if low_price < downward_start:
            first_index = bisect.bisect_left(buy_price_list, low_price)
            stop_index = bisect.bisect_left(
                buy_price_list, downward_start
            )

            for grid_index in range(stop_index - 1, first_index - 1, -1):
                if grid_index in active_by_grid:
                    continue
                if grid_index in sold_this_candle:
                    continue
                if buy_budget + tolerance < normal_order_size:
                    break

                buy_price = float(buy_prices[grid_index])
                sell_target = float(sell_targets[grid_index])
                cash_before, btc_before = cash, btc
                portfolio_value = cash_before + btc_before * buy_price

                trade_id += 1
                position = create_position(
                    trade_id=trade_id,
                    grid_index=grid_index,
                    buy_time=timestamp,
                    market_buy_price=buy_price,
                    grid_buy_price=buy_price,
                    sell_target=sell_target,
                    normal_order_size=normal_order_size,
                    portfolio_value_at_buy=portfolio_value,
                    buy_fee=buy_fee,
                    sell_fee=sell_fee,
                    entry_type="GRID_BUY",
                )

                buy_budget -= normal_order_size
                cash -= normal_order_size
                btc += position["btc_amount"]
                total_buy_fee_usdt += position["buy_fee_btc"] * buy_price

                positions[trade_id] = position
                active_by_grid[grid_index] = trade_id
                heapq.heappush(sell_heap, (sell_target, trade_id))

                event_id += 1
                events.append({
                    "event_id": event_id,
                    "time": timestamp,
                    "side": "BUY",
                    "trade_id": trade_id,
                    "grid_index": grid_index,
                    "price": buy_price,
                    "cash_movement": -normal_order_size,
                    "grid_cashflow": 0.0,
                    "cash_before": cash_before,
                    "cash_after": cash,
                    "btc_before": btc_before,
                    "btc_after": btc,
                    "portfolio_value_before": portfolio_value,
                    "initialization_trade": False,
                })

        equity_values[row_index] = cash + btc * close_price
        cash_values[row_index] = cash
        btc_values[row_index] = btc
        previous_close = close_price

    equity_curve = pd.DataFrame({
        "open_time": data["open_time"],
        "close": data["close"],
        "cash": cash_values,
        "btc": btc_values,
        "equity": equity_values,
    })

    return {
        "grid": grid,
        "initialization": initialization,
        "positions": positions,
        "trade_event_log": pd.DataFrame(events),
        "equity_curve": equity_curve,
        "final_cash": float(cash),
        "final_btc": float(btc),
        "realized_profit": float(realized_profit),
        "completed_cycles": int(completed_cycles),
        "total_buy_fee_usdt": float(total_buy_fee_usdt),
        "total_sell_fee_usdt": float(total_sell_fee_usdt),
    }


# 2. Backtest System

Historical data, benchmark, metrics, audit, and user-facing Trade History. This section does not change trading rules.


## 2.1 Backtest Configuration


In [ ]:
TIMEFRAME = "1m"
START_DATE = "2024-01-01"
END_DATE = "2026-01-01"
DATA_DIR = "/content/drive/MyDrive/03.Trading/00.Live Trading"


## 2.2 Data, Metrics & Buy/Hold Benchmark


In [ ]:
def load_market_data(symbol, timeframe, data_dir, start_date, end_date):
    path = os.path.join(data_dir, f"{symbol}-{timeframe}-combined.csv")
    if not os.path.exists(path):
        raise FileNotFoundError(path)

    data = pd.read_csv(path)
    required = {"open_time", "open", "high", "low", "close", "volume"}
    missing = required.difference(data.columns)
    if missing:
        raise ValueError(f"Missing columns: {sorted(missing)}")

    data["open_time"] = pd.to_datetime(data["open_time"], utc=True)
    numeric = ["open", "high", "low", "close", "volume"]
    data[numeric] = data[numeric].astype(float)

    start = pd.Timestamp(start_date, tz="UTC")
    end = pd.Timestamp(end_date, tz="UTC")
    data = (
        data.drop_duplicates("open_time")
        .sort_values("open_time")
        .loc[lambda x: (x["open_time"] >= start) & (x["open_time"] < end)]
        .reset_index(drop=True)
    )

    if data.empty:
        raise ValueError("No market data inside the selected period.")

    return data


def performance_stats(data, equity_curve, initial_capital):
    equity = equity_curve["equity"].to_numpy(float)
    peak = np.maximum.accumulate(equity)
    drawdown = equity / peak - 1.0

    final_equity = float(equity[-1])
    net_return = final_equity / initial_capital - 1.0
    max_drawdown = float(drawdown.min())

    elapsed_days = (
        data["open_time"].iloc[-1] - data["open_time"].iloc[0]
    ).total_seconds() / 86400.0

    annualized_return = np.nan
    if elapsed_days > 0 and final_equity > 0:
        annualized_log_growth = (
            np.log(final_equity / initial_capital)
            * (365.25 / elapsed_days)
        )
        if annualized_log_growth < 700:
            annualized_return = float(
                np.expm1(annualized_log_growth)
            )

    calmar_ratio = np.nan
    if max_drawdown < 0 and np.isfinite(annualized_return):
        calmar_ratio = float(annualized_return / abs(max_drawdown))

    equity_curve = equity_curve.copy()
    equity_curve["drawdown"] = drawdown

    return {
        "final_equity": final_equity,
        "net_return": float(net_return),
        "annualized_return": annualized_return,
        "max_drawdown": max_drawdown,
        "calmar_ratio": calmar_ratio,
        "equity_curve": equity_curve,
    }


def build_buy_hold_benchmark(data, initial_capital, buy_fee):
    entry_price = float(data["open"].iloc[0])
    gross_btc = initial_capital / entry_price
    fee_btc = gross_btc * buy_fee
    net_btc = gross_btc - fee_btc

    curve = pd.DataFrame({
        "open_time": data["open_time"],
        "close": data["close"],
        "cash": 0.0,
        "btc": net_btc,
        "equity": net_btc * data["close"],
    })

    stats = performance_stats(data, curve, initial_capital)

    return {
        "entry_price": entry_price,
        "net_btc": float(net_btc),
        "entry_fee_usdt_equiv": float(fee_btc * entry_price),
        **stats,
    }


## 2.3 Trade History & Audit


In [ ]:
def build_trade_history(positions, final_time, final_price):
    rows = []

    for trade_id in sorted(positions):
        position = positions[trade_id]
        closed = position["status"] == "CLOSED"

        if closed:
            sell_time = position["sell_time"]
            portfolio_at_sell = position["portfolio_value_at_sell"]
            net_pnl = position["net_pnl"]
            holding_time = sell_time - position["buy_time"]
        else:
            sell_time = pd.NaT
            portfolio_at_sell = np.nan
            net_pnl = (
                position["btc_amount"] * final_price
                - position["order_size_usdt"]
            )
            holding_time = final_time - position["buy_time"]

        rows.append({
            "Trade ID": int(position["trade_id"]),
            "Status": position["status"],
            "Buy Time": position["buy_time"],
            "Buy Price": float(position["buy_price"]),
            "Sell Target": float(position["sell_target"]),
            "Sell Time": sell_time,
            "Order Size (USDT)": float(position["order_size_usdt"]),
            "Portfolio Value at Buy": float(
                position["portfolio_value_at_buy"]
            ),
            "Order % of Portfolio": float(
                position["order_pct_of_portfolio"] * 100.0
            ),
            "Portfolio Value at Sell": (
                float(portfolio_at_sell)
                if np.isfinite(portfolio_at_sell)
                else np.nan
            ),
            "Net P&L": float(net_pnl),
            "Holding Time": holding_time,
        })

    return pd.DataFrame(rows)


def audit_v0b(result, trade_history, initial_capital, buy_fee):
    init = result["initialization"]
    events = result["trade_event_log"]
    equity = result["equity_curve"]
    positions = result["positions"]
    grid = result["grid"]

    cash_movement = float(events["cash_movement"].sum()) if len(events) else 0.0
    grid_cashflow = float(events["grid_cashflow"].sum()) if len(events) else 0.0

    open_btc = float(sum(
        p["btc_amount"]
        for p in positions.values()
        if p["status"] == "OPEN"
    ))

    seed_quantity_ok = True
    normal_order_size = float(init["normal_order_size_usdt"])
    for position in positions.values():
        if position["entry_type"] != "INITIAL_SEED":
            continue

        grid_buy_price = float(
            grid.loc[position["grid_index"], "buy_price"]
        )
        expected_btc = (
            normal_order_size / grid_buy_price * (1.0 - buy_fee)
        )
        if not np.isclose(
            position["btc_amount"],
            expected_btc,
            rtol=0.0,
            atol=1e-12,
        ):
            seed_quantity_ok = False
            break

    equity_error = float(np.max(np.abs(
        equity["cash"]
        + equity["btc"] * equity["close"]
        - equity["equity"]
    )))

    return {
        "cash_reconciliation": np.isclose(
            initial_capital + cash_movement,
            result["final_cash"],
            atol=1e-8,
        ),
        "realized_profit_reconciliation": np.isclose(
            grid_cashflow,
            result["realized_profit"],
            atol=1e-8,
        ),
        "equity_identity": equity_error <= 1e-8,
        "cash_never_negative": float(equity["cash"].min()) >= -1e-8,
        "initial_allocation_reconciliation": np.isclose(
            init["initial_cash"] + init["initial_btc_cost_usdt"],
            initial_capital,
            atol=1e-8,
        ),
        "all_grid_slots_initialized_or_reserved": (
            init["initial_sell_positions"]
            + init["initial_buy_levels"]
            == len(grid)
        ),
        "initial_seed_quantity_matches_normal_grid": seed_quantity_ok,
        "final_btc_matches_open_positions": np.isclose(
            result["final_btc"],
            open_btc,
            atol=1e-10,
        ),
        "closed_trade_count_reconciliation": (
            int(trade_history["Status"].eq("CLOSED").sum())
            == result["completed_cycles"]
        ),
    }


## 2.4 Run Backtest & Review Results


In [ ]:
df_1m = load_market_data(
    SYMBOL,
    TIMEFRAME,
    DATA_DIR,
    START_DATE,
    END_DATE,
)

result = run_initialized_fixed_grid(
    df_1m,
    GRID_TEMPLATE,
    INITIAL_CAPITAL,
    BUY_FEE,
    SELL_FEE,
)

stats = performance_stats(
    df_1m,
    result["equity_curve"],
    INITIAL_CAPITAL,
)
result["equity_curve"] = stats["equity_curve"]

buy_hold = build_buy_hold_benchmark(
    df_1m,
    INITIAL_CAPITAL,
    BUY_FEE,
)

final_time = df_1m["open_time"].iloc[-1]
final_price = float(df_1m["close"].iloc[-1])
trade_history = build_trade_history(
    result["positions"],
    final_time,
    final_price,
)

open_positions = int(trade_history["Status"].eq("OPEN").sum())
unrealized_pnl = float(
    trade_history.loc[
        trade_history["Status"].eq("OPEN"),
        "Net P&L",
    ].sum()
)

summary = {
    "initial_capital": float(INITIAL_CAPITAL),
    "initial_market_price": float(result["initialization"]["start_price"]),
    "normal_order_size_usdt": float(
        result["initialization"]["normal_order_size_usdt"]
    ),
    "initial_cash": float(result["initialization"]["initial_cash"]),
    "initial_btc": float(result["initialization"]["initial_btc"]),
    "initial_sell_positions": int(
        result["initialization"]["initial_sell_positions"]
    ),
    "initial_buy_levels": int(
        result["initialization"]["initial_buy_levels"]
    ),
    "final_equity": stats["final_equity"],
    "net_return": stats["net_return"],
    "annualized_return": stats["annualized_return"],
    "max_drawdown": stats["max_drawdown"],
    "calmar_ratio": stats["calmar_ratio"],
    "completed_cycles": int(result["completed_cycles"]),
    "open_positions": open_positions,
    "final_cash": float(result["final_cash"]),
    "final_btc": float(result["final_btc"]),
    "realized_profit": float(result["realized_profit"]),
    "unrealized_pnl": unrealized_pnl,
    "total_fee_usdt_equiv": float(
        result["total_buy_fee_usdt"]
        + result["total_sell_fee_usdt"]
    ),
}

benchmark_summary = {
    key: buy_hold[key]
    for key in [
        "entry_price",
        "net_btc",
        "entry_fee_usdt_equiv",
        "final_equity",
        "net_return",
        "annualized_return",
        "max_drawdown",
        "calmar_ratio",
    ]
}

comparison_vs_buy_hold = {
    "final_equity_difference_usdt": float(
        summary["final_equity"] - benchmark_summary["final_equity"]
    ),
    "excess_return": float(
        summary["net_return"] - benchmark_summary["net_return"]
    ),
    "drawdown_improvement": float(
        abs(benchmark_summary["max_drawdown"])
        - abs(summary["max_drawdown"])
    ),
    "calmar_difference": float(
        summary["calmar_ratio"] - benchmark_summary["calmar_ratio"]
    ),
}

audit_checks = audit_v0b(
    result,
    trade_history,
    INITIAL_CAPITAL,
    BUY_FEE,
)
AUDIT_STATUS = (
    "PASS"
    if all(bool(value) for value in audit_checks.values())
    else "FAIL"
)

comparison_table = pd.DataFrame({
    "Metric": [
        "Final Equity (USDT)",
        "Net Return",
        "Annualized Return",
        "Max Drawdown",
        "Calmar Ratio",
    ],
    "V0-B Grid": [
        summary["final_equity"],
        summary["net_return"],
        summary["annualized_return"],
        summary["max_drawdown"],
        summary["calmar_ratio"],
    ],
    "BTC Buy & Hold": [
        benchmark_summary["final_equity"],
        benchmark_summary["net_return"],
        benchmark_summary["annualized_return"],
        benchmark_summary["max_drawdown"],
        benchmark_summary["calmar_ratio"],
    ],
})

print("===== V0-B INITIALIZATION =====")
for key, value in result["initialization"].items():
    print(f"{key:32s}: {value}")

print("\n===== V0-B RESULT =====")
for key, value in summary.items():
    print(f"{key:32s}: {value}")

print("\n===== V0-B vs BTC BUY & HOLD =====")
display(comparison_table)
print(
    f"Excess Return vs Buy & Hold : "
    f"{comparison_vs_buy_hold['excess_return']:.4%}"
)
print(
    f"Drawdown Improvement       : "
    f"{comparison_vs_buy_hold['drawdown_improvement']:.4%}"
)

print("\n===== V0-B AUDIT =====")
for name, passed in audit_checks.items():
    print(f"{name:44s}: {'PASS' if passed else 'FAIL'}")
print(f"Overall{'':37s}: {AUDIT_STATUS}")

if AUDIT_STATUS != "PASS":
    raise AssertionError("V0-B AUDIT FAILED")


## 2.5 User Trade History


In [ ]:
display(trade_history)


# 3. Logging System

Each run creates new immutable files under `logs/v0/<run_id>/`. Existing logs are never overwritten.


## 3.1 Run Log


In [ ]:
REPO = "natdanaiii/Trading"
BRANCH = "main"

RUN_TIMESTAMP = pd.Timestamp.now(tz="UTC")
RUN_ID = RUN_TIMESTAMP.strftime("%Y%m%dT%H%M%S%fZ")
GITHUB_RUN_DIR = f"logs/v0/{RUN_ID}"

GITHUB_SUMMARY_LOG_PATH = f"{GITHUB_RUN_DIR}/summary.json"
GITHUB_TRADE_HISTORY_PATH = f"{GITHUB_RUN_DIR}/trade_history.csv"

LOCAL_SUMMARY_LOG_PATH = f"/content/{RUN_ID}_summary.json"
LOCAL_TRADE_HISTORY_PATH = f"/content/{RUN_ID}_trade_history.csv"


def json_safe(value):
    if value is pd.NaT or value is pd.NA:
        return None
    if isinstance(value, dict):
        return {str(k): json_safe(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_safe(v) for v in value]
    if isinstance(value, np.ndarray):
        return [json_safe(v) for v in value.tolist()]
    if isinstance(value, (bool, np.bool_)):
        return bool(value)
    if isinstance(value, (int, np.integer)):
        return int(value)
    if isinstance(value, (float, np.floating)):
        return None if not np.isfinite(value) else float(value)
    if isinstance(value, pd.Timestamp):
        return value.isoformat()
    return value


log_payload = json_safe({
    "log_schema_version": 5,
    "strategy": "V0-B Initialized Fixed Grid (Grid-Consistent Seed Sizing)",
    "run_info": {
        "run_id": RUN_ID,
        "generated_at_utc": RUN_TIMESTAMP.isoformat(),
        "repository": REPO,
        "branch": BRANCH,
        "notebook": "Grid_trading_V0.ipynb",
        "symbol": SYMBOL,
        "timeframe": TIMEFRAME,
        "start_date": START_DATE,
        "end_date": END_DATE,
        "data_rows": len(df_1m),
        "data_first_time": df_1m["open_time"].min(),
        "data_last_time": df_1m["open_time"].max(),
    },
    "trading_config": {
        "initial_capital": INITIAL_CAPITAL,
        "floor": GRID_FLOOR,
        "ceiling": GRID_CEILING,
        "gap": GRID_GAP,
        "buy_fee": BUY_FEE,
        "sell_fee": SELL_FEE,
    },
    "backtest_config": {
        "timeframe": TIMEFRAME,
        "start_date": START_DATE,
        "end_date": END_DATE,
    },
    "derived": {
        "number_of_grids": NUMBER_OF_GRIDS,
        "normal_order_size_usdt": (
            result["initialization"]["normal_order_size_usdt"]
        ),
    },
    "initialization": result["initialization"],
    "market_range_diagnostics": {
        "historical_low": float(df_1m["low"].min()),
        "historical_high": float(df_1m["high"].max()),
        "candles_low_below_floor": int(
            (df_1m["low"] < GRID_FLOOR).sum()
        ),
        "candles_high_above_ceiling": int(
            (df_1m["high"] > GRID_CEILING).sum()
        ),
    },
    "summary": summary,
    "buy_hold_benchmark": benchmark_summary,
    "comparison_vs_buy_hold": comparison_vs_buy_hold,
    "audit": {
        "status": AUDIT_STATUS,
        "checks": audit_checks,
    },
    "trade_history_file": GITHUB_TRADE_HISTORY_PATH,
})

with open(LOCAL_SUMMARY_LOG_PATH, "w", encoding="utf-8") as file:
    json.dump(log_payload, file, indent=2, allow_nan=False)

trade_history.to_csv(LOCAL_TRADE_HISTORY_PATH, index=False)

print(f"Run ID        : {RUN_ID}")
print(f"Summary       : {GITHUB_SUMMARY_LOG_PATH}")
print(f"Trade History : {GITHUB_TRADE_HISTORY_PATH}")


## 3.2 Upload Immutable Run Logs


In [ ]:
def create_github_file(
    path,
    content_bytes,
    commit_message,
    github_token,
    max_retries=3,
):
    url = f"https://api.github.com/repos/{REPO}/contents/{path}"
    headers = {
        "Authorization": f"Bearer {github_token}",
        "Accept": "application/vnd.github+json",
        "X-GitHub-Api-Version": "2022-11-28",
    }
    body = {
        "message": commit_message,
        "content": base64.b64encode(content_bytes).decode(),
        "branch": BRANCH,
    }

    for attempt in range(1, max_retries + 1):
        existing = requests.get(url, headers=headers, timeout=30)

        if existing.status_code == 200:
            raise FileExistsError(
                f"Immutable run log already exists: {path}"
            )
        if existing.status_code != 404:
            existing.raise_for_status()

        response = requests.put(
            url,
            headers=headers,
            json=body,
            timeout=30,
        )

        if response.status_code in (200, 201):
            return response.json()["commit"]["sha"]

        if response.status_code == 409 and attempt < max_retries:
            wait_seconds = 1.5 * attempt
            print(
                f"GitHub conflict for {path}; "
                f"retrying in {wait_seconds:.1f}s..."
            )
            time.sleep(wait_seconds)
            continue

        raise RuntimeError(
            f"GitHub upload failed [{response.status_code}] "
            f"for {path}: {response.text}"
        )


try:
    from google.colab import userdata
    github_token = userdata.get("GITHUB_TOKEN")
except Exception:
    github_token = None

if not github_token:
    print(
        "GitHub upload SKIPPED: "
        "Colab Secret 'GITHUB_TOKEN' was not found."
    )
else:
    with open(LOCAL_SUMMARY_LOG_PATH, "rb") as file:
        summary_commit = create_github_file(
            GITHUB_SUMMARY_LOG_PATH,
            file.read(),
            f"Add V0-B run {RUN_ID} summary",
            github_token,
        )

    with open(LOCAL_TRADE_HISTORY_PATH, "rb") as file:
        history_commit = create_github_file(
            GITHUB_TRADE_HISTORY_PATH,
            file.read(),
            f"Add V0-B run {RUN_ID} trade history",
            github_token,
        )

    print("GitHub run-log upload: SUCCESS")
    print(f"Summary Commit : {summary_commit}")
    print(f"History Commit : {history_commit}")
